In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{Definition} $$

$$ \mathbb{R}^n \to \mathbb{R}, \quad p_{i}(z) = \frac{e^{z_i}}{\sum_{k} e^{z_k}} $$

$$ \mathbb{R}^n \to \mathbb{R}^n, \quad p(z) = (p_1(z), p_2(z), \ldots, p_n(z)) $$

$$ \text{Derivative} $$

$$
\frac{dp_i}{dz_j} = p_i (\delta_{ij} - p_j) =
\begin{dcases}
    i = j & p_i (1 - p_i) \\
    i \neq j & p_i (0 - p_j)\\
\end{dcases}
$$

$$ \text{Jacobian} $$

$$
J_p(z) =
\text{diag}(p) - p p^\top =
\begin{cases}
    i = j & p_i (1 - p_i) \\
    i \neq j & -p_i p_j\\
\end{cases}
$$

$$ dp = J_p(z) \, dz $$

$$ \text{Frobenius equality} $$

$$
\left \langle \frac{dF}{dz}, dz \right \rangle =
\left \langle \frac{dF}{dp}, dp \right \rangle
$$

$$ \\[2em] $$

$$
\left \langle \frac{dF}{dp}, dp \right \rangle =
$$

$$
\left \langle \frac{dF}{dp}, J_p(z) \, dz \right \rangle =
$$

$$
\left \langle \frac{dF}{dp}, \Big(\text{diag}(p) - p p^\top \Big) \, dz \right \rangle =
$$

$$
\left \langle \Big(\text{diag}(p) - p p^\top \Big)^\top \, \frac{dF}{dp}, dz \right \rangle
$$

$$
\left \langle \Big(\text{diag}(p) - p p^\top \Big) \, \frac{dF}{dp}, dz \right \rangle
$$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore


def _softmax(z: tr.Tensor, dim: 1) -> tr.Tensor:
    """
        Numerically stable implementation of the `softmax`.
        For simplicity, only the case of `dim=1` is supported.
    """

    assert dim == 1

    ex = tr.exp(z)
    return ex / tr.sum(ex, dim=-1, keepdim=True)


def softmax(z: tr.Tensor, dim: int = 1) -> tr.Tensor:
    """
        Numerically stable implementation of the `softmax`.
        For simplicity, only the case of `dim=1` is supported.
    """

    assert dim == 1

    z = tr.as_tensor(z)
    return _softmax(z - tr.max(z, dim=-1, keepdim=True).values, dim=1)


class SoftmaxFunction(autograd.Function):
    """Custom autograd function for the `softmax`."""

    @staticmethod
    def forward(ctx, z: tr.Tensor) -> tr.Tensor:
        p = softmax(z, dim=1)
        ctx.save_for_backward(p)
        return p

    @staticmethod
    def backward(ctx, dF_dp: tr.Tensor) -> tuple[tr.Tensor,]:
        (p, ) = ctx.saved_tensors

        dot = (dF_dp * p).sum(dim=1, keepdim=True)
        dF_dz = p * (dF_dp - dot)

        return (dF_dz, )


class Softmax(nn.Module):
    """Custom module for the `softmax`."""

    def __init__(self) -> None:
        super().__init__()

    def forward(self, z: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(z)

In [ ]:
# Unit tests

def test_basic() -> None:
    assert softmax(1000, dim=1) == approx(1.0)
    assert softmax(100, dim=1) == approx(1.0)
    assert softmax(10, dim=1) == approx(1.0)
    assert softmax(1, dim=1) == approx(1.0)


def test_gradcheck(samples) -> None:
    def func(z: tr.Tensor) -> tr.Tensor:
        return SoftmaxFunction.apply(z)

    z = tr.randn(samples, 3, dtype=tr.float64, requires_grad=True)
    assert autograd.gradcheck(func, (z, ), eps=0.001, atol=0.001)


def test_compare(samples) -> None:
    z = tr.randn(samples, 3, dtype=tr.float32, requires_grad=True)

    z1 = z.clone().detach().requires_grad_(True)
    p1 = Softmax()(z1)
    p1.sum().backward()

    z2 = z.clone().detach().requires_grad_(True)
    p2 = tr.softmax(z2, dim=-1)
    p2.sum().backward()

    assert p1 == approx(p2)
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_basic()

    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)